# 11.3 CPU 与 GPU 训练

> **模块十一 · PyTorch深度学习实践** | 第6课时
>
> **学习目标：**
> 1. 理解 CPU vs GPU 在深度学习中的差异
> 2. 掌握 `torch.device` 的使用
> 3. 学会将数据和模型移至 GPU
> 4. 了解混合精度训练的基本概念
> 5. 了解多 GPU 训练（DDP）的概念

---

## 1. CPU vs GPU 在深度学习中的差异

### 架构对比

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心数 | 少（4-64个） | 多（数千个） |
| 单核性能 | 强 | 较弱 |
| 内存带宽 | ~50 GB/s | ~1000 GB/s |
| 适合任务 | 串行、复杂逻辑 | 大规模并行计算 |
| 显存 | 系统内存（大） | 显存（通常 8-24 GB） |

**深度学习为什么用 GPU？**
- 矩阵乘法、卷积等运算是**高度并行**的
- GPU 的**数千个核心**可以同时处理大量数据
- GPU 的**高带宽显存**加速数据搬运

**典型加速比：** GPU 训练深度学习模型通常比 CPU 快 **10-100 倍**。

---
## 2. CUDA 环境检查

CUDA 是 NVIDIA 提供的 GPU 并行计算平台。PyTorch 通过 CUDA 调用 GPU 资源。

In [1]:
import torch

print("=" * 50)
print("CUDA 环境检查")
print("=" * 50)

# PyTorch 是否支持 CUDA
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"cuDNN 版本: {torch.backends.cudnn.version()}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  显存: {props.total_memory / 1024**3:.1f} GB")
        print(f"  计算能力: {props.major}.{props.minor}")
        print(f"  多处理器数量: {props.multi_processor_count}")
else:
    print("\n⚠️ CUDA 不可用，将使用 CPU 进行演示")
    print("如需 GPU 支持，请确保:")
    print("  1. 安装了 NVIDIA 显卡驱动")
    print("  2. 安装了 CUDA Toolkit")
    print("  3. 安装了 GPU 版本的 PyTorch")

CUDA 环境检查
PyTorch 版本: 1.13.1+cu117
CUDA 是否可用: True
CUDA 版本: 11.7
cuDNN 版本: 8500
GPU 数量: 1

GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU
  显存: 8.0 GB
  计算能力: 8.9
  多处理器数量: 24


---
## 3. `torch.device` — 设备管理

`torch.device` 用于指定张量/模型所在的设备。

In [2]:
# ========== 设备选择 ==========

# 方式1: 手动指定
device_cpu = torch.device('cpu')
print(f"CPU 设备: {device_cpu}")

# 方式2: 自动选择 (推荐)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"自动选择设备: {device}")

# 方式3: 指定具体 GPU
if torch.cuda.is_available():
    device_0 = torch.device('cuda:0')  # 第一块 GPU
    device_1 = torch.device('cuda:1')  # 第二块 GPU
    print(f"GPU 0: {device_0}")
    print(f"GPU 1: {device_1}")

# 获取当前设备
print(f"\n当前默认设备: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

CPU 设备: cpu
自动选择设备: cuda
GPU 0: cuda:0
GPU 1: cuda:1

当前默认设备: cuda


---
## 4. 数据和模型的 `.to(device)`

### 4.1 张量移至 GPU

In [3]:
# ========== 张量的设备操作 ==========

# CPU 上的张量
x_cpu = torch.randn(3, 3)
print(f"x_cpu 设备: {x_cpu.device}")

# 移至 GPU
if torch.cuda.is_available():
    x_gpu = x_cpu.to('cuda')
    print(f"x_gpu 设备: {x_gpu.device}")
    
    # 多种写法等价
    x_gpu2 = x_cpu.cuda()
    x_gpu3 = x_cpu.to(device)
    
    # 直接在 GPU 上创建
    y_gpu = torch.randn(3, 3, device='cuda')
    z_gpu = torch.zeros(3, 3, device=device)
    
    print(f"\n直接在GPU上创建的张量设备: {y_gpu.device}")
    
    # GPU 间移动
    # y_gpu_to_1 = y_gpu.to('cuda:1')  # 移到 GPU 1
else:
    print("\n⚠️ 当前无 GPU，以下在 CPU 上演示")
    x_gpu = x_cpu.to('cpu')  # fallback
    print(f"x_gpu 设备: {x_gpu.device}")

x_cpu 设备: cpu
x_gpu 设备: cuda:0

直接在GPU上创建的张量设备: cuda:0


In [4]:
# ⚠️ 设备不匹配会报错!
# a = torch.randn(3, 3).cpu()
# b = torch.randn(3, 3).cuda()
# c = a + b  # RuntimeError: Expected all tensors to be on the same device

print("⚠️ 注意: 不同设备上的张量不能直接运算!")
print("必须将两者移至同一设备后再计算。")

⚠️ 注意: 不同设备上的张量不能直接运算!
必须将两者移至同一设备后再计算。


### 4.2 模型移至 GPU

模型移至 GPU 后，其所有参数和缓冲区都会移至 GPU。

In [5]:
import torch.nn as nn

# 定义模型
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = DemoModel()

# 检查模型参数所在设备
def get_model_device(model):
    return next(model.parameters()).device

print(f"模型初始设备: {get_model_device(model)}")

# 移至 GPU
model = model.to(device)
print(f"模型当前设备: {get_model_device(model)}")

# 检查每个参数的设备
print("\n各层参数设备:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.device}")

模型初始设备: cpu
模型当前设备: cuda:0

各层参数设备:
  fc1.weight: cuda:0
  fc1.bias: cuda:0
  fc2.weight: cuda:0
  fc2.bias: cuda:0


### 4.3 GPU 训练的标准模板

将之前的训练循环改为 GPU 版本，只需修改几行代码：

In [6]:
print("""GPU 训练模板:

# 1. 定义设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 模型移至 GPU
model = MyNet().to(device)

# 3. 训练循环中，数据和标签移至 GPU
for images, labels in train_loader:
    images = images.to(device)    # ← 关键!
    labels = labels.to(device)    # ← 关键!
    
    outputs = model(images)
    loss = criterion(outputs, labels)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
""")

GPU 训练模板:

# 1. 定义设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 模型移至 GPU
model = MyNet().to(device)

# 3. 训练循环中，数据和标签移至 GPU
for images, labels in train_loader:
    images = images.to(device)    # ← 关键!
    labels = labels.to(device)    # ← 关键!
    
    outputs = model(images)
    loss = criterion(outputs, labels)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



---
## 5. GPU 加速演示

### 5.1 矩阵乘法速度对比

In [7]:
import time

def benchmark_matmul(size, device, n_iter=100):
    """矩阵乘法性能测试"""
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # 预热
    for _ in range(10):
        _ = a @ b
    
    if device.type == 'cuda':
        torch.cuda.synchronize()  # 等待 GPU 完成
    
    start = time.time()
    for _ in range(n_iter):
        _ = a @ b
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return elapsed

# ========== 性能对比 ==========
sizes = [512, 1024, 2048, 4096]
n_iter = 50

print(f"矩阵乘法性能对比 (迭代 {n_iter} 次):\n")
print(f"{'矩阵大小':>10s} | {'CPU (秒)':>10s} | {'GPU (秒)':>10s} | {'加速比':>8s}")
print("-" * 50)

for size in sizes:
    cpu_time = benchmark_matmul(size, torch.device('cpu'), n_iter)
    
    if torch.cuda.is_available():
        gpu_time = benchmark_matmul(size, torch.device('cuda'), n_iter)
        speedup = cpu_time / gpu_time
        print(f"{size:>10d} | {cpu_time:>10.3f} | {gpu_time:>10.3f} | {speedup:>7.1f}x")
    else:
        print(f"{size:>10d} | {cpu_time:>10.3f} | {'N/A':>10s} | {'N/A':>8s}")

矩阵乘法性能对比 (迭代 50 次):

      矩阵大小 |    CPU (秒) |    GPU (秒) |      加速比
--------------------------------------------------
       512 |      0.169 |      0.004 |    44.6x
      1024 |      0.608 |      0.020 |    30.0x
      2048 |      3.909 |      0.138 |    28.3x
      4096 |     27.602 |      0.961 |    28.7x


### 5.2 模型训练速度对比

对比同一个模型在 CPU 和 GPU 上的训练速度。

In [8]:
import torch.nn as nn
import torch.optim as optim

# ========== 定义较大的模型用于测试 ==========
class BenchmarkNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 16 * 16, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def benchmark_training(device_name, n_batches=50):
    """训练性能测试"""
    device = torch.device(device_name)
    
    model = BenchmarkNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 生成模拟数据
    images = torch.randn(32, 3, 64, 64, device=device)
    labels = torch.randint(0, 10, (32,), device=device)
    
    # 预热
    for _ in range(5):
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.time()
    for _ in range(n_batches):
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    throughput = n_batches * 32 / elapsed  # images/sec
    return elapsed, throughput


# ========== 运行对比 ==========
print(f"模型训练速度对比 (batch_size=32, {50} iterations):\n")
print(f"{'设备':>8s} | {'总耗时(秒)':>12s} | {'吞吐量(img/s)':>15s}")
print("-" * 45)

cpu_time, cpu_throughput = benchmark_training('cpu', n_batches=20)
print(f"{'CPU':>8s} | {cpu_time:>12.3f} | {cpu_throughput:>15.1f}")

if torch.cuda.is_available():
    gpu_time, gpu_throughput = benchmark_training('cuda', n_batches=50)
    print(f"{'GPU':>8s} | {gpu_time:>12.3f} | {gpu_throughput:>15.1f}")
    print(f"\n🚀 GPU 加速比: {cpu_time / gpu_time:.1f}x")
else:
    print(f"{'GPU':>8s} | {'N/A':>12s} | {'N/A':>15s}")

模型训练速度对比 (batch_size=32, 50 iterations):

      设备 |       总耗时(秒) |      吞吐量(img/s)
---------------------------------------------
     CPU |       11.077 |            57.8
     GPU |        1.736 |           921.8

🚀 GPU 加速比: 6.4x


---
## 6. 混合精度训练 (Mixed Precision)

### 6.1 基本概念

混合精度训练的核心思想是：
- 部分计算使用 **FP16（半精度）** 加速
- 部分计算保持 **FP32（全精度）** 保证精度

| 精度类型 | 位数 | 范围 | 显存占用 |
|----------|------|------|----------|
| FP32 | 32位 | ±3.4×10³⁸ | 1x |
| FP16 | 16位 | ±6.5×10⁴ | 0.5x |

**优点：**
- 训练速度提升 1.5-3x（在支持 Tensor Core 的 GPU 上）
- 显存占用减半，可以增大 batch size

**关键技术：**
- **Loss Scaling**：将 loss 乘以一个大数（如 2¹⁶），避免 FP16 梯度下溢
- **FP32 Master Weights**：维护一份 FP32 的权重副本用于更新

In [9]:
# ========== 混合精度训练 (使用 torch.cuda.amp) ==========
print("""混合精度训练模板:

from torch.cuda.amp import autocast, GradScaler

# 创建 GradScaler
scaler = GradScaler()

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    
    optimizer.zero_grad()
    
    # 自动混合精度前向传播
    with autocast():
        outputs = model(images)
        loss = criterion(outputs, labels)
    
    # 缩放梯度后反向传播
    scaler.scale(loss).backward()
    
    # 更新参数
    scaler.step(optimizer)
    scaler.update()
""")

# PyTorch 2.0+ 推荐写法
print("""
PyTorch 2.0+ 更简洁的写法:

scaler = torch.amp.GradScaler('cuda')

with torch.amp.autocast('cuda'):
    outputs = model(images)
    loss = criterion(outputs, labels)
    
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
""")

混合精度训练模板:

from torch.cuda.amp import autocast, GradScaler

# 创建 GradScaler
scaler = GradScaler()

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    
    optimizer.zero_grad()
    
    # 自动混合精度前向传播
    with autocast():
        outputs = model(images)
        loss = criterion(outputs, labels)
    
    # 缩放梯度后反向传播
    scaler.scale(loss).backward()
    
    # 更新参数
    scaler.step(optimizer)
    scaler.update()


PyTorch 2.0+ 更简洁的写法:

scaler = torch.amp.GradScaler('cuda')

with torch.amp.autocast('cuda'):
    outputs = model(images)
    loss = criterion(outputs, labels)
    
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()



In [10]:
# ========== 混合精度 vs 全精度速度对比 ==========
if torch.cuda.is_available():
    device = torch.device('cuda')
    model = BenchmarkNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    images = torch.randn(64, 3, 64, 64, device=device)
    labels = torch.randint(0, 10, (64,), device=device)
    
    n_iter = 100
    
    # FP32
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_iter):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    fp32_time = time.time() - start
    
    # FP16 (混合精度)
    scaler = torch.amp.GradScaler('cuda')
    
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_iter):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize()
    fp16_time = time.time() - start
    
    print(f"FP32 训练耗时: {fp32_time:.3f}s")
    print(f"FP16 混合精度耗时: {fp16_time:.3f}s")
    print(f"混合精度加速比: {fp32_time/fp16_time:.2f}x")
else:
    print("⚠️ 需要 GPU 才能运行混合精度对比")

AttributeError: module 'torch.amp' has no attribute 'GradScaler'

---
## 7. 多 GPU 训练概念

当单块 GPU 显存不够或想加速训练时，可以使用多 GPU 训练。

### 7.1 主要方式

| 方式 | API | 特点 |
|------|-----|------|
| **DataParallel (DP)** | `nn.DataParallel` | 简单但有性能瓶颈 |
| **DistributedDataParallel (DDP)** | `torch.nn.parallel.DistributedDataParallel` | ✅ 推荐，高效 |

### 7.2 DataParallel (DP)

```python
model = nn.DataParallel(model, device_ids=[0, 1])
model = model.to('cuda')
```

工作方式：
- GPU 0 作为主 GPU，负责收集梯度和更新参数
- 数据被自动切分到各 GPU
- **缺点**：GPU 0 负担重，通信开销大

### 7.3 DistributedDataParallel (DDP)

```python
# 每个 GPU 运行一个独立进程
model = DDP(model, device_ids=[local_rank])
```

**优点：**
- 每个 GPU 有独立的优化器和数据
- 使用 All-Reduce 进行梯度同步，更高效
- 支持多机多卡

**启动方式：**
```bash
torchrun --nproc_per_node=4 train.py
```

In [11]:
# ========== DataParallel 简单示例 ==========
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    model = DemoModel()
    model = nn.DataParallel(model)  # 自动使用所有 GPU
    model = model.to(device)
    print(f"DataParallel 模型: 使用 {torch.cuda.device_count()} 块 GPU")
else:
    print(f"当前 GPU 数量: {torch.cuda.device_count()}")
    print("DataParallel 需要多块 GPU")

当前 GPU 数量: 1
DataParallel 需要多块 GPU


---
## 8. 常见问题与最佳实践

### ❌ 常见错误

1. **设备不匹配**：模型在 GPU，数据在 CPU
   ```python
   # 错误!
   model = model.to('cuda')
   for x, y in loader:  # x, y 在 CPU
       output = model(x)  # RuntimeError!
   ```

2. **忘记同步**：GPU 异步执行，计时需用 `torch.cuda.synchronize()`

3. **忘记将标签移至 GPU**：`labels = labels.to(device)`

### ✅ 最佳实践

1. **统一设备管理**：
   ```python
   device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
   ```

2. **数据移至设备**：在 DataLoader 中用 `pin_memory=True` 加速 CPU→GPU 传输

3. **清理显存**：训练完成后用 `torch.cuda.empty_cache()`

4. **监控显存**：
   ```python
   print(f"显存使用: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
   print(f"显存缓存: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
   ```

In [12]:
# ========== GPU 显存监控 ==========
if torch.cuda.is_available():
    print("GPU 显存信息:")
    print(f"  已分配: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  缓存中: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    print(f"  最大分配: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
    
    # nvidia-smi
    print("\n运行 nvidia-smi 查看详细信息:")
    !nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader 2>/dev/null || echo "nvidia-smi 不可用"
else:
    print("⚠️ 无 GPU，无法显示显存信息")

GPU 显存信息:
  已分配: 0.51 GB
  缓存中: 1.32 GB
  最大分配: 0.99 GB

运行 nvidia-smi 查看详细信息:
"nvidia-smi ������"

ϵͳ�Ҳ���ָ����·����


---
## 9. 本节小结

| 主题 | 要点 |
|------|------|
| 设备选择 | `torch.device('cuda' if torch.cuda.is_available() else 'cpu')` |
| 张量移至GPU | `tensor.to(device)` 或 `tensor.cuda()` |
| 模型移至GPU | `model.to(device)`（所有参数自动移至GPU） |
| 训练模板 | 数据和模型必须在同一设备 |
| 混合精度 | `torch.amp.autocast()` + `GradScaler()` |
| 多GPU | DataParallel(简单) / DDP(推荐) |
| 计时注意 | GPU 异步执行，需 `torch.cuda.synchronize()` |

---

## 📝 练习题

### 练习1: 设备管理

编写一个函数 `train_on_device(device_name)`，接受设备字符串参数（`'cpu'` 或 `'cuda'`），完成以下任务：
1. 在指定设备上创建一个 3 层 MLP
2. 生成随机数据（1000个样本，每个10维特征，3分类）
3. 训练 50 个 epoch，记录每 epoch 的训练时间
4. 返回总训练时间

分别调用 `'cpu'` 和 `'cuda'`（如果有 GPU），对比训练时间。

### 练习2: 显存监控

创建一个简单的 CNN，观察不同 batch size 下的显存占用：
- batch_size = 16, 32, 64, 128, 256
- 记录每种 batch_size 下的显存占用
- 绘制 batch_size vs 显存占用的关系图

**提示：** 使用 `torch.cuda.memory_allocated()` 测量显存。

### 练习3: 混合精度训练

对 MNIST 分类模型分别使用 FP32 和混合精度（AMP）训练：
1. 对比每个 epoch 的训练时间
2. 对比最终的测试准确率
3. 分析精度损失情况